# 02 — Memory hierarchy and bandwidth

**Question.** How much traffic can each level of the memory system sustain, what does a dependent access cost, and how do the Triad, read-only and non-temporal-store kernels differ?

GB/s is kernel traffic (bytes the kernel reads and writes), not controller traffic. Working-set sizes are per thread for the hierarchy sweep (private arrays, first-touched by their owner) and total for the full-socket kernels.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/hwbench').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from hwbench import report, roofline, plotting
from hwbench.notebook import setup
ctx = setup('02_memory_hierarchy_and_bandwidth')
thesis, ext, theory = ctx.thesis, ctx.ext, ctx.theory
table, figure, say, say_claims = ctx.table, ctx.figure, ctx.say, ctx.say_claims
from hwbench import claims

## 1. Read bandwidth against working-set size

**Conditions.** One core and all 56 cores of socket 0, AVX-512 loads from private per-thread arrays, local memory. Hierarchy transitions appear as steps: L1 (48 KiB/core), L2 (2 MiB/core) and the shared L3 (112.5 MiB/socket). The one-core curve is the per-core capability at each level; the 56-core curve is the socket aggregate.

In [ ]:
h = report.hierarchy(ext)
one = h[h.notes == 'hier_read_1']
fig = plotting.line_by_config(one, 'per_thread_bytes', 'gbs', 'config', 'Read bandwidth · one core',
                              'Working set per core (bytes)', 'GB/s', log_y=True)
plotting.mark_x(fig, {'L1d 48 KiB': 48 * 2**10, 'L2 2 MiB': 2 * 2**20, 'L3 112.5 MiB': 112.5 * 2**20})
plotting.bytes_axis(fig)
figure(fig, 'hierarchy_read_1core')

In [ ]:
full = h[h.notes == 'hier_read_56']
fig = plotting.line_by_config(full, 'per_thread_bytes', 'gbs', 'config', 'Read bandwidth · 56 cores, one socket',
                              'Working set per core (bytes)', 'GB/s', log_y=True)
plotting.mark_x(fig, {'L1d 48 KiB': 48 * 2**10, 'L2 / L3 share 2 MiB': 2 * 2**20})
plotting.bytes_axis(fig, 2**14, 2**28)
figure(fig, 'hierarchy_read_56cores')

In [ ]:
levels, latency = report.level_summary(h)
table(levels.pivot(index='level', columns='config', values='gbs').reset_index())

In [ ]:
lv = levels.pivot(index='level', columns='config', values='gbs')
say(f"**Result.** Per core, L1 sustains {lv.loc['L1 (32 KiB)','flat_hbm']:.0f} GB/s, L2 {lv.loc['L2 (1 MiB)','flat_hbm']:.0f} GB/s and the L3 "
    f"{lv.loc['LLC (16 MiB)','flat_hbm']:.0f} GB/s; a single core reaches only {lv.loc['memory, 1 core (4 GiB)','flat_hbm']:.0f} GB/s from memory. "
    f"All 56 cores aggregate to {lv.loc['L1, 56 cores (32 KiB/core)','flat_hbm']/1000:.1f} TB/s in L1 and {lv.loc['L2, 56 cores (1 MiB/core)','flat_hbm']/1000:.1f} TB/s in L2, "
    f"against {lv.loc['memory, 56 cores (256 MiB/core)','flat_hbm']:.0f} GB/s (HBM) and {lv.loc['memory, 56 cores (256 MiB/core)','flat_ddr']:.0f} GB/s (DDR) from memory.")

In [ ]:
say_claims(claims.hierarchy(ext, theory))

## 2. Idle dependent-load latency

**Conditions.** One core chasing a random cycle of 64-byte nodes (seed 7); transparent huge pages are enabled system-wide, which keeps translation cheap up to a few GiB but not beyond (claim `latency_translation` below). Results describe idle latency, not loaded latency.

In [ ]:
lat = h[h.notes == 'hier_latency']
fig = plotting.line_by_config(lat, 'per_thread_bytes', 'latency_ns', 'config', 'Dependent-load latency · one core',
                              'Working set (bytes)', 'ns per load', log_y=True)
plotting.mark_x(fig, {'L1d': 48 * 2**10, 'L2': 2 * 2**20, 'L3': 112.5 * 2**20})
plotting.bytes_axis(fig)
figure(fig, 'hierarchy_latency')

In [ ]:
table(latency.pivot(index='level', columns='config', values='latency_ns').reset_index())

The cache-mode latency of a miss was measured by chasing a cycle of 16 to 128 GiB (a one-core pointer chase; building and walking the 128 GiB cycle takes minutes).

In [ ]:
far = report.cache_latency_far(ext)
near = lat[(lat.config == 'cache') & (lat.per_thread_bytes == 4 * 2**30)]
cache_lat = pd.concat([pd.DataFrame({'working_set_gib': [4.0], 'latency_ns': [float(near.latency_ns.iloc[0])]}), far[['working_set_gib', 'latency_ns']]], ignore_index=True)
mem_lat = latency.pivot(index='level', columns='config', values='latency_ns').loc['memory']
fig = plotting.cache_latency_figure(cache_lat, {'DDR': float(mem_lat['flat_ddr']), 'HBM': float(mem_lat['flat_hbm'])})
figure(fig, 'cache_latency_far')
table(far, 1)
say_claims(claims.latency(ext, theory, ROOT), ['latency_order', 'cache_latency_far', 'latency_translation'])

Cache levels are identical across configurations; only beyond the L3 do tiers differ. HBM's benefit is bandwidth under many outstanding requests, not latency.

## 3. Triad, read-only and non-temporal Triad

**Conditions.** Socket 0, 56 cores, 8 GiB total, local memory (Flat), Cache mode at 8 GiB (HBM-resident). Triad moves 24 B per element as counted; if the hardware also read the destination line (write-allocate) the bus carries 32 B, so a Triad that is not write-allocate-free would read lower than the non-temporal variant, which never reads the destination.

In [ ]:
bw = report.bandwidth_summary(ext, thesis)
local = bw[(bw.locality == 'local') & (bw.socket_scope == 'single') & (bw.socket == 0) & (bw.working_set_gib == 8)]
fig = plotting.go.Figure()
for cfg in ['flat_ddr', 'flat_hbm', 'cache']:
    part = local[local.config == cfg].groupby('kernel').gbs.mean()
    fig.add_bar(x=list(part.index), y=list(part.values), name=plotting.CONFIG_LABELS[cfg], marker_color=plotting.CONFIG_COLORS[cfg])
plotting.style(fig, 'Sustained bandwidth by kernel · one socket, 56 cores, 8 GiB', 'Kernel', 'GB/s', 480)
fig.update_layout(barmode='group')
figure(fig, 'kernel_comparison')
table(local.pivot_table(index='config', columns='kernel', values='gbs').reset_index())

In [ ]:
p = local.pivot_table(index='config', columns='kernel', values='gbs')
say(f"**Result.** Triad and non-temporal Triad agree to {100*abs(p.loc['flat_ddr','triad']/p.loc['flat_ddr','triad_nt']-1):.1f}% on DDR ({p.loc['flat_ddr','triad']:.0f} and {p.loc['flat_ddr','triad_nt']:.0f} GB/s) "
    f"and {100*abs(p.loc['flat_hbm','triad']/p.loc['flat_hbm','triad_nt']-1):.1f}% on HBM ({p.loc['flat_hbm','triad']:.0f} and {p.loc['flat_hbm','triad_nt']:.0f} GB/s), "
    f"so the destination line is not being read before it is written. Read-only is {p.loc['flat_ddr','read']/p.loc['flat_ddr','triad']-1:+.0%} on DDR ({p.loc['flat_ddr','read']:.0f} GB/s, the firmware-advertised 262) "
    f"and {p.loc['flat_hbm','read']/p.loc['flat_hbm','triad']-1:+.0%} on HBM ({p.loc['flat_hbm','read']:.0f} GB/s): reading is not faster than a mixed read/write stream on HBM.")

In [ ]:
say_claims(claims.kernels(ext, thesis, theory), ['no_write_allocate_bound', 'triad_store_width', 'read_vs_triad', 'hbm_over_ddr'])
say_claims(claims.counters(ROOT), ['counters_unprivileged', 'write_allocate_counted'])

Counted Triad bandwidth is therefore used as a proxy for bus traffic, with the caveat that this is inferred, not observed. For a roofline the primary ceiling is the best sustained of the three kernels (ERT convention); workload-matched ceilings (read-only for GEMM weights) are used where the access pattern is known.

## 4. Core count needed to saturate memory

In [ ]:
rs = report.read_scaling(ext)
rs0 = rs[rs.socket == 0]
fig = plotting.line_by_config(rs0, 'threads', 'gbs', 'config', 'Read bandwidth against core count · socket 0, 8 GiB',
                              'Cores', 'GB/s', error='std')
figure(fig, 'read_core_scaling')
sat = (rs0.groupby('config').apply(lambda g: g[g.gbs >= 0.9 * g.gbs.max()].threads.min(), include_groups=False)).rename('cores for 90% of peak').reset_index()
table(sat)

In [ ]:
say_claims(claims.core_scaling(ext, thesis, ROOT), ['ddr_saturates', 'hbm_still_scaling', 'hbm_clock_drop', 'hmat_gap'])

A workload that uses fewer cores than the saturation point (for example a server pinned to 55 cores competing with other work) leaves HBM bandwidth unused.

## 5. Cache mode against working-set size

**Conditions.** One socket, 56 cores, read-only, Triad (regular stores) and non-temporal Triad, local DDR-backed allocation of 8–128 GiB. The 64 GiB HBM capacity is a reference line, not a fitted transition.

In [ ]:
cap = bw[(bw.config == 'cache') & (bw.socket_scope == 'single') & (bw.socket == 0)]
flat_ref = bw[(bw.config != 'cache') & (bw.socket_scope == 'single') & (bw.socket == 0) & (bw.locality == 'local') & ((bw.working_set_gib - 8).abs() < 0.5)]
gm_all = report.gemm_points(ext)
gemv = gm_all[(gm_all.memory_mode == 'cache') & (gm_all['shape'] == '14336x4096') & (gm_all.m == 1)]
fig = plotting.cache_capacity_figure(cap, flat_ref, {'16 GiB: 8B BF16 model': 16, '64 GiB: HBM capacity': 64}, gemv)
figure(fig, 'cache_capacity')
table(cap.pivot_table(index='working_set_gib', columns='kernel', values='gbs').reset_index())

In [ ]:
say_claims(claims.cache_mode(ext, thesis))

Cache-mode bandwidth must be quoted with its working-set size. A model whose weights are larger than a few tens of GiB should not expect HBM-class bandwidth in Cache mode.

## 6. Can DDR and HBM bandwidth be combined? (October measurement)

In [ ]:
from hwbench.analysis import concurrent_table
table(concurrent_table(thesis).drop(columns=['cpu_bind']))

In [ ]:
say_claims(claims.concurrent(thesis))

## 7. Counted against modeled traffic

Every record of the follow-up campaigns carries the cores' own DRAM line counts (offcore-response events, 64 bytes per line, reads and writes). The points below compare them with the kernels' modeled bytes: 1 means the traffic model describes what the cores moved. The events cannot tell DDR from HBM (the NUMA binding does) and see only the measured cores' traffic.

In [ ]:
sub = ext[ext.notes.isin(['counted', 'counted_sweep', 'counted_gemm'])]
ct = report.counted_table(sub)
short = {'stream_read': 'read', 'stream_triad': 'Triad', 'stream_triad_nt': 'NT Triad'}
tiers = {'flat_ddr': 'Flat DDR', 'flat_hbm': 'Flat HBM', 'cache': 'Cache'}
ct['config'] = [plotting.config_key(m, t) for m, t in zip(ct.memory_mode, ct.memory_target)]
def name(r):
    gib = f' {r.working_set_bytes / 2**30:.0f} GiB' if r.config == 'cache' and r.benchmark != 'onednn_gemm_sweep' else (' 16 GiB pool' if r.config == 'cache' else '')
    if r.benchmark == 'roofline_sweep':
        what = f'FP64 sweep K={int(r.k)}'
    elif r.benchmark == 'onednn_gemm_sweep':
        what = f"{r['dtype'].upper()} GEMM 14336×4096 M={int(r.m)}"
    else:
        what = short[r.benchmark]
    return f'{tiers[r.config]} · {what}{gib}'
ct['label'] = ct.apply(name, axis=1)
order = {'read': 0, 'Triad': 1, 'NT Triad': 2}
ct['sort'] = [(0 if r.benchmark.startswith('stream') else 1 if r.benchmark == 'roofline_sweep' else 2, r.config, r.working_set_bytes, r.dtype if isinstance(r.dtype, str) else '', r.m if r.m == r.m else 0, r.k if r.k == r.k else 0) for r in ct.itertuples()]
ct = ct.sort_values('sort').drop(columns='sort')
fig = plotting.counted_ratio_figure(ct)
figure(fig, 'counted_vs_modeled')
table(ct[['label', 'modeled_bytes', 'counted_bytes', 'counted_over_modeled', 'read_share', 'intensity_modeled', 'intensity_counted']].assign(modeled_GB=lambda f: f.modeled_bytes / 1e9, counted_GB=lambda f: f.counted_bytes / 1e9)[['label', 'modeled_GB', 'counted_GB', 'counted_over_modeled', 'read_share', 'intensity_modeled', 'intensity_counted']], 3)
say_claims(claims.counted(ext, thesis), ['counted_streams', 'counted_gemm_weights', 'counted_gemm_prefill'])